# Deep learning on images

## Load modules from repo

In [43]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [44]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [45]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

In [46]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [47]:
import tensorflow as tf

In [48]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [49]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [50]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_y_train=y_train

In [51]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing

In [53]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [54]:
if rebalance_with_weights:
    print('using class weights')
else:
    print('not using class weights')

using class weights


In [55]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [56]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [57]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [58]:
print(X_train.shape)

(6793, 31)


In [59]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [60]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [61]:
new_preprocessors

{}

In [62]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [63]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [64]:
from tensorflow import keras

### Load or create model

In [65]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    subversion = last_experiment.get('subversion', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Reprise depuis le meilleur modèle de l'expérience : artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras


In [66]:
if not load_model:
    subversion = int(input(f"subversion (architecture)? (last: {subversion})"))

In [67]:
subversion

4

### Summary

In [68]:
# model.summary()

## Callbacks

### ModelCheckpoint

In [69]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [70]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [71]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [72]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [73]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [74]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [75]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [76]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
tensor_board_folder_timestamp = tensor_board_folder / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder_timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [77]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [78]:
# max_epochs=13

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [79]:
# Pick max_epochs based on your available time
available_minutes=60

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

29

### compilation and callbacks

In [80]:
learning_rate=0.001

In [81]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [82]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [83]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=60, max_epochs=29, champion_path='artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-04_val_loss-1.9397_f1-0.4594.keras' ?

In [84]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=max(total_epochs_trained-1,0), callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 5/34


2025-10-03 18:02:25.535410: I external/local_xla/xla/service/service.cc:163] XLA service 0x7639f00035c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-03 18:02:25.535467: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-03 18:02:25.925175: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-03 18:02:27.356023: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-03 18:02:28.432369: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 18:02:28.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.4472 - loss: 2.1491

2025-10-03 18:03:32.303879: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12664', 12 bytes spill stores, 12 bytes spill loads

2025-10-03 18:03:36.045401: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 18:03:36.139721: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 18:03:36.705187: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 18:03:36.804689: E external/local_xla/xla/s

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.4472 - loss: 2.1490

2025-10-03 18:04:53.745304: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-03 18:05:01.450397: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 18:05:01.548400: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-03 18:05:02.352752: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 171s 656ms/step - accuracy: 0.4581 - loss: 2.1242 - val_accuracy: 0.5255 - val_loss: 1.8916 - learning_rate: 0.0010
Epoch 6/34
213/213 ━━━━━━━━━━━━━━━━━━━━ 104s 488ms/step - accuracy: 0.4771 - loss: 2.0533 - val_accuracy: 0.5264 - val_loss: 1.8885 - learning_rate: 0.0010
Epoch 7/34
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 481ms/step - accuracy: 0.5004 - loss: 1.9711 - val_accuracy: 0.5367 - val_loss: 1.8789 - learning_rate: 0.0010
Epoch 8/34
213/213 ━━━━━━━━━━━━━━━━━━━━ 101s 475ms/step - accuracy: 0.5328 - loss: 1.9003 - val_accuracy: 0.5386 - val_loss: 1.8831 - learning_rate: 0.0010
Epoch 9/34
212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.5660 - loss: 1.8152
Epoch 9: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 478ms/step - accuracy: 0.5672 - loss: 1.7969 - val_accuracy: 0.5353 - val_loss: 1.9011 - learning_rate: 0.0010
Epoch 10/34
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 480ms/step - accuracy: 0.6146 -

'total_minutes=14.759327896436055'

## Evaluation

In [85]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [86]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5419217944145203,
 'actual_epochs': 12,
 'minutes_per_epoch': 1.2299439913696713}

In [87]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [88]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 71s 124ms/step


array([[5.4880977e-04, 2.9513843e-03, 7.0673209e-03, ..., 3.6099702e-02,
        6.9629552e-04, 5.9968414e-04],
       [4.9058553e-03, 1.0845932e-02, 1.7704329e-02, ..., 1.7824812e-02,
        1.3668210e-03, 5.4592029e-03],
       [1.5203415e-04, 2.1911655e-03, 2.4218010e-02, ..., 2.1687026e-01,
        1.3907533e-04, 1.5084037e-04],
       ...,
       [1.9178626e-01, 1.2115647e-02, 3.0512692e-05, ..., 2.9396286e-05,
        6.5318483e-01, 4.5857360e-03],
       [2.8534611e-03, 7.1947570e-03, 9.5198853e-03, ..., 4.0990572e-02,
        1.1880313e-03, 8.8156416e-04],
       [4.8211184e-03, 1.6134378e-02, 2.2002023e-02, ..., 6.3045159e-02,
        5.6349435e-03, 1.9567930e-03]], shape=(16984, 27), dtype=float32)

In [89]:
from sklearn import metrics

In [90]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [91]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1280,1281,1300,1302,1320,1560,1920,2060,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,
10,166,20,0,0,1,5,4,0,2,0,2,1,2,5,86,173,1,9,0,6,0,140,0
40,17,198,1,8,9,18,3,1,49,0,3,3,1,10,29,28,15,6,0,20,0,80,3
50,0,19,31,27,9,5,14,0,104,1,2,6,1,11,1,10,12,13,0,66,0,1,3
60,0,6,6,97,0,4,2,0,29,0,0,0,0,1,2,2,10,1,0,3,0,1,2
1140,4,38,0,1,301,11,71,0,13,1,9,0,1,9,5,7,3,3,0,41,0,13,3
1160,4,48,0,0,10,673,1,0,0,0,0,0,0,0,15,24,6,0,0,3,0,7,0
1180,4,15,0,0,38,8,20,0,5,0,4,1,0,6,6,17,4,4,0,14,0,6,1
1280,1,26,4,4,82,5,373,1,204,6,33,13,11,71,3,18,2,18,0,90,2,4,3
1281,3,42,0,3,13,34,79,2,21,2,7,4,0,34,8,52,15,24,0,36,0,25,10


In [92]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.49081755972123103, 0.4594376012796143)

In [93]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [94]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

,precision,recall,f1-score,support
10,0.486804,0.266453,0.344398,623.00000
40,0.359347,0.394422,0.376068,502.00000
50,0.469697,0.092262,0.154229,336.00000
60,0.591463,0.584337,0.587879,166.00000
1140,0.555351,0.563670,0.559480,534.00000
1160,0.810843,0.850822,0.830352,791.00000
1180,0.000000,0.000000,0.000000,153.00000
1280,0.366405,0.382957,0.374498,974.00000
1281,0.500000,0.004831,0.009569,414.00000
1300,0.491793,0.801784,0.609646,1009.00000


In [95]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.445192,0.410292,0.392017,629.037037
std,0.225169,0.315290,0.270646,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.362876,0.066211,0.113086,310.000000
50%,0.491793,0.394422,0.444898,534.000000
75%,0.574647,0.668665,0.604440,953.500000
max,0.810843,0.895201,0.830352,2042.000000


In [96]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.715634,0.816498,0.073989
recall,0.715634,1.000000,0.969083,0.106320
f1-score,0.816498,0.969083,1.000000,0.094113
support,0.073989,0.106320,0.094113,1.000000


In [97]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.49081755972123103

In [98]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5419217944145203,
 'actual_epochs': 12,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.49081755972123103,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.27064638775487243)}

## Update tracker

In [99]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmin(model_history.history['val_loss'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_loss = model_history.history['val_loss'][best_epoch_in_session_idx]


    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epoch_index-{best_epoch_global:02d}_val_loss-{best_val_loss:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même subversion {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras
Effacement de l'ancien modèle de la même subversion {champion_path=} .


In [100]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

In [101]:
to_track=['subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5419217944145203,
 'actual_epochs': 12,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.49081755972123103,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.27064638775487243),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras',
 'epoch_index': np.int64(7),
 'total_epochs': np.int64(13),
 'subversion': 4,
 'max_epochs': 29,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 32,
  'image_dense': 64,
  'tabular_dense_2': 16,
  'final_dense_1': 128},
 'embedding_dims': {'pHash_embedding': 8, 'md5_embedding': 8}}

In [107]:
tracker['comment']="Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep."
tracker['comment']

'Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.'

In [108]:
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.5419217944145203,
 'actual_epochs': 12,
 'minutes_per_epoch': 1.9549971238772073,
 'weighted_avg_f1_score': 0.49081755972123103,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.27064638775487243),
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras',
 'epoch_index': np.int64(7),
 'total_epochs': np.int64(13),
 'subversion': 4,
 'max_epochs': 29,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 32,
  'image_dense': 64,
  'tabular_dense_2': 16,
  'final_dense_1': 128},
 'embedding_dims': {'pHash_embedding': 8, 'md5_embedding': 8},
 'comment': 'Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.'}

In [104]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [105]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [106]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [109]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, log_file_path=log_file_path)

Log pour l'expérience subversion 4 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [110]:
pd.set_option('max_colwidth', None)

In [111]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment,best_model_path
0,1,False,6793,32,2.024143,9,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which indicates overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
1,2,False,6793,32,1.998601,19,10,0.001,0.569713,0.528593,0.0,0.247884,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
2,2,False,20380,32,3.867331,9,8,0.001,0.584609,0.563841,0.0,0.239801,256_128_64_32,16_16,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras
3,3,False,6793,32,2.024143,9,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
4,4,True,6793,32,1.954997,29,13,0.001,0.541922,0.490818,0.0,0.270646,128_64_32_16,8_8,"Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.",artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.


In [ ]:
print(tracking_df)

   subversion  rebalance_with_weights  X_train.shape[0]  BATCH_SIZE  \
0           1                   False             20380          32   
1           2                   False              6793          32   
2           2                   False             20380          32   

   minutes_per_epoch  max_epochs  total_epochs  learning_rate  val_accuracy  \
0           2.657786           8            13          0.001      0.608220   
1           1.998601          19            10          0.001      0.569713   
2           3.867331           9             8          0.001      0.584609   

   weighted_avg_f1_score  min_f1_score  std_f1_score dense_layers_sizes  \
0               0.598100      0.254545      0.180580      256_128_64_32   
1               0.528593      0.000000      0.247884      256_128_64_32   
2               0.563841      0.000000      0.239801      256_128_64_32   

  embedding_dims  \
0          16_16   
1          16_16   
2          16_16   

                

In [ ]:
tracking_df.drop(columns=['best_model_path','minutes_per_epoch'])

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,max_epochs,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,False,20380,32,8,13,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
1,2,False,6793,32,19,10,0.001,0.569713,0.528593,0.000000,0.247884,256_128_64_32,16_16,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.
2,2,False,20380,32,9,8,0.001,0.584609,0.563841,0.000000,0.239801,256_128_64_32,16_16,Increased frac from 0.1 to 0.3.
